<a href="https://www.kaggle.com/code/tommasofacchin/01-resident-evil-data-scraping-and-generation?scriptVersionId=305052703" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import numpy as np 
import pandas as pd 
import requests
from pathlib import Path
from bs4 import BeautifulSoup

# Resident Evil: data scraping and generation

Resident Evil game universe dataset created by web scraping and AI generation, intended as a base for future projects.

# Datesets

**Game** *(id, title, year, type, chronology_order)*: AI Generated

**Character** *(id, name, role)*: Resident evil 4 characters scraped from wikipedia, the rest AI generated.

**GameAppearence** *(gameId, characterId, role)*

**Dialogues** *(id, game_id, scene_id, line_index, character_id, text, timestamp, source)*: TODO

**Scenes** *(id, game_id, name, sequence_index)*: TODO


## Games

In [2]:
games_data = [
    # id, title, year, type, chronology_order
    (1,  "Resident Evil",                             1996, "mainline", 2),
    (2,  "Resident Evil 2",                           1998, "mainline", 4),
    (3,  "Resident Evil 3: Nemesis",                  1999, "mainline", 3),
    (4,  "Resident Evil: Code – Veronica",            2000, "mainline", 5),
    (5,  "Resident Evil 0",                           2002, "mainline", 1),
    (6,  "Resident Evil 4",                           2005, "mainline", 6),
    (7,  "Resident Evil 5",                           2009, "mainline", 8),
    (8,  "Resident Evil: Revelations",                2012, "spinoff",  7),
    (9,  "Resident Evil 6",                           2012, "mainline", 10),
    (10, "Resident Evil: Revelations 2",              2015, "spinoff",  9),
    (11, "Resident Evil 7: Biohazard",                2017, "mainline", 11),
    (12, "Resident Evil Village",                     2021, "mainline", 12),
    (13, "Resident Evil Village – Shadow of Rose",    2022, "spinoff",  13),
    (14, "Resident Evil Requiem",                     2026, "mainline", 14),
]

games = pd.DataFrame(
    games_data,
    columns=["id", "title", "year", "type", "chronology_order"]
)

games.to_csv("games.csv", index=False)
games.head(20)

,id,title,year,type,chronology_order
0,1,Resident Evil,1996,mainline,2
1,2,Resident Evil 2,1998,mainline,4
2,3,Resident Evil 3: Nemesis,1999,mainline,3
3,4,Resident Evil: Code – Veronica,2000,mainline,5
4,5,Resident Evil 0,2002,mainline,1
5,6,Resident Evil 4,2005,mainline,6
6,7,Resident Evil 5,2009,mainline,8
7,8,Resident Evil: Revelations,2012,spinoff,7
8,9,Resident Evil 6,2012,mainline,10
9,10,Resident Evil: Revelations 2,2015,spinoff,9


## Characters and GameAppearence

In [3]:
characters = pd.DataFrame(
    columns=[
        "id",       
        "name",    
        "role",     
        #"faction",  
    ]
)

gameAppearance = pd.DataFrame(
    columns=[
        "gameId",       
        "characterId", 
        "role",        
    ]
)

In [4]:
# Resident evil 4 (2005): easy scraping from wikipedia
game_id = 6
URL = "https://it.wikipedia.org/wiki/Resident_Evil_4"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

resp = requests.get(URL, headers=headers)
soup = BeautifulSoup(resp.text, "html.parser")

# Patch: Kaggle runs notebook offline during save and run, so i downloaded the wikipedia page in order for it to work.
html_path = "/kaggle/input/datasets/tommasofacchin/resident-evil-4-wikipedia/Resident Evil 4 - Wikipedia.html" 

with open(html_path, "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")
# -- end patch


names = []
roles = [] 
current_role = None 

for tag in soup.find_all(["h2", "h3"]):
    if tag.name == "h2":
        text = tag.get_text(strip=True)
        print("H2:", text)  

        if text == "Personaggi":
            current_role = "hero"
        elif text == "Nemici":
            current_role = "villain"
        else:
            current_role = None

    elif tag.name == "h3" and current_role is not None:
        text = tag.get_text(strip=True)

        names.append(text)
        roles.append(current_role)
        print("  H3:", text)

print("\n# characters:", len(names))
print(list(zip(names, roles)))


re4_df = pd.DataFrame({
    "name": names,
    "role_in_game": roles,  
})

# Characters
re4_df.loc[re4_df["name"] == "Albert Wesker", "role_in_game"] = "villain"
existing_names = set(characters["name"])

missing_mask = ~re4_df["name"].isin(existing_names)
missing_rows = re4_df[missing_mask]

next_id = characters["id"].max() + 1 if len(characters) > 0 else 1

new_rows = []
for _, row in missing_rows.iterrows():
    new_rows.append({
        "id": next_id,
        "name": row["name"],
        "role": row["role_in_game"],
    })
    next_id += 1

if new_rows:
    new_chars = pd.DataFrame(new_rows)
    characters = pd.concat([characters, new_chars], ignore_index=True)

# Game appearences
rows = []
for _, row in re4_df.iterrows():
    name = row["name"]
    role_in_game = row["role_in_game"]

    match = characters[characters["name"] == name]
    if match.empty:
        continue
    cid = match["id"].iloc[0]

    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re4 = pd.DataFrame(rows)

gameAppearance = pd.concat([gameAppearance, gameApp_re4], ignore_index=True)

print(f"\n\ncharacters: {characters}")
print(f"\n\nGame appearence: {gameAppearance}")

H2: Indice
H2: Trama
H2: Modalità di gioco
H2: Personaggi
  H3: Leon Scott Kennedy
  H3: Ashley Graham
  H3: Ada Wong
  H3: Ingrid Hunnigan
  H3: Luis Sera
  H3: Albert Wesker
  H3: HUNK
  H3: Mercante d'armi
H2: Nemici
  H3: Las Plagas
  H3: Los Ganados
  H3: Lord Osmund Saddler
  H3: Ramon Salazar
  H3: Bitores Mendez
  H3: Jack Krauser
H2: Versioni
H2: Confronti territoriali
H2: Versione beta
H2: Remake
H2: Edizione VR
H2: Accoglienza
H2: Doppiaggio
H2: Colonna sonora
H2: Note
H2: Altri progetti
H2: Collegamenti esterni

# characters: 14
[('Leon Scott Kennedy', 'hero'), ('Ashley Graham', 'hero'), ('Ada Wong', 'hero'), ('Ingrid Hunnigan', 'hero'), ('Luis Sera', 'hero'), ('Albert Wesker', 'hero'), ('HUNK', 'hero'), ("Mercante d'armi", 'hero'), ('Las Plagas', 'villain'), ('Los Ganados', 'villain'), ('Lord Osmund Saddler', 'villain'), ('Ramon Salazar', 'villain'), ('Bitores Mendez', 'villain'), ('Jack Krauser', 'villain')]


characters:     id                 name     role
0    1   Leon

In [5]:
# Resident evil (1996): AI generated
game_id = 1 
next_id = characters["id"].max() + 1 

character_data = [
    # name,                 main_alignment
    ("Chris Redfield",      "hero"),
    ("Jill Valentine",      "hero"),
    ("Albert Wesker",       "villain"),
    ("Barry Burton",        "hero"),
    ("Brad Vickers",        "support"),
    ("Joseph Frost",        "support"),
    ("Rebecca Chambers",    "support"),
    ("Richard Aiken",       "support"),
    ("Enrico Marini",       "support"),
    ("Forest Speyer",       "support"),
    ("Kenneth J. Sullivan", "support"),
]
existing_names = set(characters["name"])

new_rows = []
for name, role in character_data:
    if name in existing_names:           
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role, 
    })
    next_id += 1

new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])

characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))
gameAppearance_data = [
    # name,                 role_in_game
    ("Chris Redfield",      "hero"),
    ("Jill Valentine",      "hero"),
    ("Albert Wesker",       "villain"),
    ("Barry Burton",        "support"),
    ("Brad Vickers",        "support"),
    ("Joseph Frost",        "support"),
    ("Rebecca Chambers",    "support"),
    ("Richard Aiken",       "support"),
    ("Enrico Marini",       "support"),
    ("Forest Speyer",       "support"),
    ("Kenneth J. Sullivan", "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re1 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re1], ignore_index=True)


print(f"\n\ncharacters: {characters}")
print(f"\n\nGame appearence: {gameAppearance}")



characters:     id                 name     role
0    1   Leon Scott Kennedy     hero
1    2        Ashley Graham     hero
2    3             Ada Wong     hero
3    4      Ingrid Hunnigan     hero
4    5            Luis Sera     hero
5    6        Albert Wesker  villain
6    7                 HUNK     hero
7    8      Mercante d'armi     hero
8    9           Las Plagas  villain
9   10          Los Ganados  villain
10  11  Lord Osmund Saddler  villain
11  12        Ramon Salazar  villain
12  13       Bitores Mendez  villain
13  14         Jack Krauser  villain
14  15       Chris Redfield     hero
15  16       Jill Valentine     hero
16  17         Barry Burton     hero
17  18         Brad Vickers  support
18  19         Joseph Frost  support
19  20     Rebecca Chambers  support
20  21        Richard Aiken  support
21  22        Enrico Marini  support
22  23        Forest Speyer  support
23  24  Kenneth J. Sullivan  support


Game appearence:    gameId characterId     role
0       6  

In [6]:
# Resident evil 2 (1998): AI generated
game_id = 2
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Leon Scott Kennedy", "hero"),      
    ("Claire Redfield",    "hero"),
    ("Ada Wong",           "hero"),      
    ("Sherry Birkin",      "support"),
    ("William Birkin",     "villain"),
    ("Annette Birkin",     "support"),
    ("Brian Irons",        "villain"),
    ("Marvin Branagh",     "support"),
    ("Ben Bertolucci",     "support"),
    ("Robert Kendo",       "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue 
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Leon Scott Kennedy", "hero"),       
    ("Claire Redfield",    "hero"),        
    ("Ada Wong",           "support"),     
    ("Sherry Birkin",      "support"),
    ("William Birkin",     "villain"),
    ("Annette Birkin",     "support"),
    ("Brian Irons",        "villain"),
    ("Marvin Branagh",     "support"),
    ("Ben Bertolucci",     "support"),
    ("Robert Kendo",       "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re2 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re2], ignore_index=True)
print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == 2])

    id                name     role
0    1  Leon Scott Kennedy     hero
2    3            Ada Wong     hero
24  25     Claire Redfield     hero
25  26       Sherry Birkin  support
26  27      William Birkin  villain
27  28      Annette Birkin  support
28  29         Brian Irons  villain
29  30      Marvin Branagh  support
30  31      Ben Bertolucci  support
31  32        Robert Kendo  support
   gameId characterId     role
25      2           1     hero
26      2          25     hero
27      2           3  support
28      2          26  support
29      2          27  villain
30      2          28  support
31      2          29  villain
32      2          30  support
33      2          31  support
34      2          32  support


In [7]:
# Resident evil 3: Nemesis (1999): AI generated
game_id = 3
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Jill Valentine",   "hero"),      
    ("Carlos Oliveira",  "hero"),
    ("Nemesis",          "villain"),
    ("Nikolai Zinoviev", "villain"),
    ("Mikhail Victor",   "support"),
    ("Brad Vickers",     "support"),   
    ("Marvin Branagh",   "support"),
    ("Dario Rosso",      "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Jill Valentine",   "hero"),       
    ("Carlos Oliveira",  "hero"),       
    ("Nemesis",          "villain"), 
    ("Nikolai Zinoviev", "villain"),
    ("Mikhail Victor",   "support"),
    ("Brad Vickers",     "support"),
    ("Marvin Branagh",   "support"),
    ("Dario Rosso",      "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re3 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re3], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == 3])

    id              name     role
15  16    Jill Valentine     hero
17  18      Brad Vickers  support
29  30    Marvin Branagh  support
32  33   Carlos Oliveira     hero
33  34           Nemesis  villain
34  35  Nikolai Zinoviev  villain
35  36    Mikhail Victor  support
36  37       Dario Rosso  support
   gameId characterId     role
35      3          16     hero
36      3          33     hero
37      3          34  villain
38      3          35  villain
39      3          36  support
40      3          18  support
41      3          30  support
42      3          37  support


In [8]:
# Resident evil: Code – Veronica (2000): AI generated
game_id = 4
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Claire Redfield",  "hero"),
    ("Steve Burnside",   "hero"),
    ("Chris Redfield",   "hero"),
    ("Albert Wesker",    "villain"),
    ("Alexia Ashford",   "villain"),
    ("Alfred Ashford",   "villain"),
    ("Rodrigo Juan Raval","support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Claire Redfield",  "hero"),        
    ("Steve Burnside",   "hero"),       
    ("Chris Redfield",   "hero"),       
    ("Albert Wesker",    "villain"), 
    ("Alexia Ashford",   "villain"), 
    ("Alfred Ashford",   "villain"),
    ("Rodrigo Juan Raval","support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_cvx = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_cvx], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

    id                name     role
5    6       Albert Wesker  villain
14  15      Chris Redfield     hero
24  25     Claire Redfield     hero
37  38      Steve Burnside     hero
38  39      Alexia Ashford  villain
39  40      Alfred Ashford  villain
40  41  Rodrigo Juan Raval  support
   gameId characterId     role
43      4          25     hero
44      4          38     hero
45      4          15     hero
46      4           6  villain
47      4          39  villain
48      4          40  villain
49      4          41  support


In [9]:
# Resident evil 0 (2002): AI generated
game_id = 5
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Rebecca Chambers", "hero"),
    ("Billy Coen",       "hero"),
    ("James Marcus",     "villain"),
    ("Edward Dewey",     "support"),
    ("Enrico Marini",    "support"),
    ("Albert Wesker",    "villain"),
    ("William Birkin",   "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Rebecca Chambers", "hero"),       
    ("Billy Coen",       "hero"),        
    ("James Marcus",     "villain"),  
    ("Edward Dewey",     "support"),
    ("Enrico Marini",    "support"),
    ("Albert Wesker",    "villain"),
    ("William Birkin",   "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re0 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re0], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])


    id              name     role
5    6     Albert Wesker  villain
19  20  Rebecca Chambers  support
21  22     Enrico Marini  support
26  27    William Birkin  villain
41  42        Billy Coen     hero
42  43      James Marcus  villain
43  44      Edward Dewey  support
   gameId characterId     role
50      5          20     hero
51      5          42     hero
52      5          43  villain
53      5          44  support
54      5          22  support
55      5           6  villain
56      5          27  villain


In [10]:
# Resident evil 5 (2009): AI generated
game_id = 7
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Chris Redfield",  "hero"),
    ("Sheva Alomar",    "hero"),
    ("Albert Wesker",   "villain"),
    ("Jill Valentine",  "hero"),
    ("Ricardo Irving",  "villain"),
    ("Excella Gionne",  "villain"),
    ("Josh Stone",      "support"),
    ("Ozwell E. Spencer","villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Chris Redfield",   "hero"),        
    ("Sheva Alomar",     "hero"),        
    ("Albert Wesker",    "villain"),  
    ("Jill Valentine",   "support"),     
    ("Ricardo Irving",   "villain"),
    ("Excella Gionne",   "villain"),
    ("Josh Stone",       "support"),
    ("Ozwell E. Spencer","support"),     
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re5 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re5], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

    id               name     role
5    6      Albert Wesker  villain
14  15     Chris Redfield     hero
15  16     Jill Valentine     hero
44  45       Sheva Alomar     hero
45  46     Ricardo Irving  villain
46  47     Excella Gionne  villain
47  48         Josh Stone  support
48  49  Ozwell E. Spencer  villain
   gameId characterId     role
57      7          15     hero
58      7          45     hero
59      7           6  villain
60      7          16  support
61      7          46  villain
62      7          47  villain
63      7          48  support
64      7          49  support


In [11]:
# Resident Evil: Revelations (2012): AI generated
game_id = 8
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Jill Valentine",   "hero"),
    ("Chris Redfield",   "hero"),
    ("Parker Luciani",   "hero"),
    ("Jessica Sherawat", "hero"),    
    ("Clive R. O'Brian", "support"),
    ("Raymond Vester",   "support"),  
    ("Morgan Lansdale",  "villain"),
    ("Jack Norman",      "villain"),
    ("Rachel Foley",     "support"),
    ("Keith Lumley",     "support"),
    ("Quint Cetcham",    "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Jill Valentine",   "hero"),
    ("Chris Redfield",   "hero"),
    ("Parker Luciani",   "support"),
    ("Jessica Sherawat", "support"),
    ("Clive R. O'Brian", "support"),
    ("Raymond Vester",   "support"),    
    ("Morgan Lansdale",  "villain"), 
    ("Jack Norman",      "villain"),
    ("Rachel Foley",     "support"),
    ("Keith Lumley",     "support"),
    ("Quint Cetcham",    "support"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_rev = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_rev], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

    id              name     role
14  15    Chris Redfield     hero
15  16    Jill Valentine     hero
49  50    Parker Luciani     hero
50  51  Jessica Sherawat     hero
51  52  Clive R. O'Brian  support
52  53    Raymond Vester  support
53  54   Morgan Lansdale  villain
54  55       Jack Norman  villain
55  56      Rachel Foley  support
56  57      Keith Lumley  support
57  58     Quint Cetcham  support
   gameId characterId     role
65      8          16     hero
66      8          15     hero
67      8          50  support
68      8          51  support
69      8          52  support
70      8          53  support
71      8          54  villain
72      8          55  villain
73      8          56  support
74      8          57  support
75      8          58  support


In [12]:
# Resident evil 6 (2012): AI generated
game_id = 9
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Leon Scott Kennedy", "hero"),
    ("Chris Redfield",     "hero"),
    ("Jake Muller",        "hero"),
    ("Ada Wong",           "hero"),      
    ("Sherry Birkin",      "support"),
    ("Helena Harper",      "hero"),
    ("Piers Nivans",       "hero"),
    ("Ingrid Hunnigan",    "support"),
    ("Derek C. Simmons",   "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Leon Scott Kennedy", "hero"),        
    ("Helena Harper",      "hero"),        
    ("Chris Redfield",     "hero"),       
    ("Piers Nivans",       "hero"),        
    ("Jake Muller",        "hero"),       
    ("Sherry Birkin",      "support"),    
    ("Ada Wong",           "support"),    
    ("Ingrid Hunnigan",    "support"),
    ("Derek C. Simmons",   "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re6 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re6], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

    id                name     role
0    1  Leon Scott Kennedy     hero
2    3            Ada Wong     hero
3    4     Ingrid Hunnigan     hero
14  15      Chris Redfield     hero
25  26       Sherry Birkin  support
58  59         Jake Muller     hero
59  60       Helena Harper     hero
60  61        Piers Nivans     hero
61  62    Derek C. Simmons  villain
   gameId characterId     role
76      9           1     hero
77      9          60     hero
78      9          15     hero
79      9          61     hero
80      9          59     hero
81      9          26  support
82      9           3  support
83      9           4  support
84      9          62  villain


In [13]:
# Resident Evil: Revelations 2 (2015): AI generated
game_id = 10
next_id = characters["id"].max() + 1 

character_data = [
    # name,            main_alignment
    ("Claire Redfield", "hero"),
    ("Barry Burton",    "hero"),
    ("Moira Burton",    "support"),
    ("Natalia Korda",   "support"),
    ("Alex Wesker",     "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,            role_in_game
    ("Claire Redfield", "hero"),
    ("Barry Burton",    "hero"),
    ("Moira Burton",    "support"),
    ("Natalia Korda",   "support"),
    ("Alex Wesker",     "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_rev2 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_rev2], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

    id             name     role
16  17     Barry Burton     hero
24  25  Claire Redfield     hero
62  63     Moira Burton  support
63  64    Natalia Korda  support
64  65      Alex Wesker  villain
   gameId characterId     role
85     10          25     hero
86     10          17     hero
87     10          63  support
88     10          64  support
89     10          65  villain


In [14]:
# Resident Evil 7: Biohazard (2017): AI generated
game_id = 11
next_id = characters["id"].max() + 1 

character_data = [
    # name,           main_alignment
    ("Ethan Winters", "hero"),
    ("Mia Winters",   "hero"),
    ("Jack Baker",    "villain"),
    ("Marguerite Baker","villain"),
    ("Lucas Baker",   "villain"),
    ("Zoe Baker",     "support"),
    ("Eveline",       "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,           role_in_game
    ("Ethan Winters", "hero"),
    ("Mia Winters",   "support"),     
    ("Jack Baker",    "villain"),
    ("Marguerite Baker","villain"),
    ("Lucas Baker",   "villain"),
    ("Zoe Baker",     "support"),
    ("Eveline",       "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_re7 = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_re7], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

    id              name     role
65  66     Ethan Winters     hero
66  67       Mia Winters     hero
67  68        Jack Baker  villain
68  69  Marguerite Baker  villain
69  70       Lucas Baker  villain
70  71         Zoe Baker  support
71  72           Eveline  villain
   gameId characterId     role
90     11          66     hero
91     11          67  support
92     11          68  villain
93     11          69  villain
94     11          70  villain
95     11          71  support
96     11          72  villain


In [15]:
# Resident Evil Village (2021): AI generated
game_id = 12
next_id = characters["id"].max() + 1 

character_data = [
    # name,               main_alignment
    ("Ethan Winters",     "hero"),
    ("Mia Winters",       "hero"),
    ("Rosemary Winters",  "support"),
    ("Chris Redfield",    "hero"),
    ("Mother Miranda",    "villain"),
    ("Alcina Dimitrescu", "villain"),
    ("Karl Heisenberg",   "villain"),
    ("Salvatore Moreau",  "villain"),
    ("Donna Beneviento",  "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,               role_in_game
    ("Ethan Winters",     "hero"),
    ("Mia Winters",       "support"),
    ("Rosemary Winters",  "support"),
    ("Chris Redfield",    "support"),
    ("Mother Miranda",    "villain"),
    ("Alcina Dimitrescu", "villain"),
    ("Karl Heisenberg",   "villain"),
    ("Salvatore Moreau",  "villain"),
    ("Donna Beneviento",  "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_village = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_village], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

    id               name     role
14  15     Chris Redfield     hero
65  66      Ethan Winters     hero
66  67        Mia Winters     hero
72  73   Rosemary Winters  support
73  74     Mother Miranda  villain
74  75  Alcina Dimitrescu  villain
75  76    Karl Heisenberg  villain
76  77   Salvatore Moreau  villain
77  78   Donna Beneviento  villain
    gameId characterId     role
97      12          66     hero
98      12          67  support
99      12          73  support
100     12          15  support
101     12          74  villain
102     12          75  villain
103     12          76  villain
104     12          77  villain
105     12          78  villain


In [16]:
# Resident Evil Village – Shadow of Rose (2022): AI generated
game_id = 13
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Rosemary Winters", "hero"),
    ("Ethan Winters",    "hero"),
    ("Mother Miranda",   "villain"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Rosemary Winters", "hero"),
    ("Ethan Winters",    "support"),
    ("Mother Miranda",   "villain"),
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_sor = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_sor], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

    id              name     role
65  66     Ethan Winters     hero
72  73  Rosemary Winters  support
73  74    Mother Miranda  villain
    gameId characterId     role
106     13          73     hero
107     13          66  support
108     13          74  villain


In [17]:
# Resident Evil Requiem (2026): AI generated
game_id = 14
next_id = characters["id"].max() + 1 

character_data = [
    # name,              main_alignment
    ("Grace Ashcroft",   "hero"),
    ("Leon Scott Kennedy","hero"),
    ("Sherry Birkin",    "support"),
    ("Victor Gideon",    "villain"),
    ("Nathan Dempsey",     "support"),
    ("Alyssa Ashcroft",   "support"),
]

existing_names = set(characters["name"])
new_rows = []

for name, role in character_data:
    if name in existing_names:
        continue
    new_rows.append({
        "id": next_id,
        "name": name,
        "role": role,
    })
    next_id += 1

if new_rows:
    new_chars_df = pd.DataFrame(new_rows, columns=["id", "name", "role"])
    characters = pd.concat([characters, new_chars_df], ignore_index=True)

# Game appearence
name_to_id = dict(zip(characters["name"], characters["id"]))

gameAppearance_data = [
    # name,              role_in_game
    ("Grace Ashcroft",    "hero"),       
    ("Leon Scott Kennedy","hero"),       
    ("Sherry Birkin",     "support"),
    ("Victor Gideon",     "villain"),
    ("Nathan Dempsey",     "support"),
    ("Alyssa Ashcroft",   "support")
]

rows = []
for name, role_in_game in gameAppearance_data:
    cid = name_to_id[name]
    rows.append({
        "gameId": game_id,
        "characterId": cid,
        "role": role_in_game,
    })

gameApp_requiem = pd.DataFrame(rows, columns=["gameId", "characterId", "role"])
gameAppearance = pd.concat([gameAppearance, gameApp_requiem], ignore_index=True)

print(characters[characters["name"].isin([n for n, _ in character_data])])
print(gameAppearance[gameAppearance["gameId"] == game_id])

    id                name     role
0    1  Leon Scott Kennedy     hero
25  26       Sherry Birkin  support
78  79      Grace Ashcroft     hero
79  80       Victor Gideon  villain
80  81      Nathan Dempsey  support
81  82     Alyssa Ashcroft  support
    gameId characterId     role
109     14          79     hero
110     14           1     hero
111     14          26  support
112     14          80  villain
113     14          81  support
114     14          82  support


In [18]:
characters.to_csv("characters.csv", index=False)
print(characters.to_string())

    id                 name     role
0    1   Leon Scott Kennedy     hero
1    2        Ashley Graham     hero
2    3             Ada Wong     hero
3    4      Ingrid Hunnigan     hero
4    5            Luis Sera     hero
5    6        Albert Wesker  villain
6    7                 HUNK     hero
7    8      Mercante d'armi     hero
8    9           Las Plagas  villain
9   10          Los Ganados  villain
10  11  Lord Osmund Saddler  villain
11  12        Ramon Salazar  villain
12  13       Bitores Mendez  villain
13  14         Jack Krauser  villain
14  15       Chris Redfield     hero
15  16       Jill Valentine     hero
16  17         Barry Burton     hero
17  18         Brad Vickers  support
18  19         Joseph Frost  support
19  20     Rebecca Chambers  support
20  21        Richard Aiken  support
21  22        Enrico Marini  support
22  23        Forest Speyer  support
23  24  Kenneth J. Sullivan  support
24  25      Claire Redfield     hero
25  26        Sherry Birkin  support
2

In [19]:
gameAppearance = (
    gameAppearance
    .sort_values(by=["gameId", "characterId"])
    .reset_index(drop=True)
)

# for semplicity
gameAppearance = (
    gameAppearance
    .merge(
        games[["id", "title"]],
        left_on="gameId",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"]) 
    .rename(columns={"title": "game_title"})
    
    .merge(
        characters[["id", "name"]],
        left_on="characterId",
        right_on="id",
        how="left"
    )
    .drop(columns=["id"]) 
    .rename(columns={"name": "character_name"})
)

gameAppearance.to_csv("gameAppearance.csv", index=False)
print(gameAppearance.to_string())

    gameId characterId     role                              game_title       character_name
0        1           6  villain                           Resident Evil        Albert Wesker
1        1          15     hero                           Resident Evil       Chris Redfield
2        1          16     hero                           Resident Evil       Jill Valentine
3        1          17  support                           Resident Evil         Barry Burton
4        1          18  support                           Resident Evil         Brad Vickers
5        1          19  support                           Resident Evil         Joseph Frost
6        1          20  support                           Resident Evil     Rebecca Chambers
7        1          21  support                           Resident Evil        Richard Aiken
8        1          22  support                           Resident Evil        Enrico Marini
9        1          23  support                           Resident Evi